# AOS_LSSTCam_bps_FAM_submisstions_2026-04-15

Checking the output of `bps_submit_fam_template_20260315.yaml ` in `/sdf/group/rubin/shared/scichris/LSSTCam_FAM_reprocessing` 


    day_obs: 20260315
    computeSite: s3df
    pipelineYaml: $DONUT_VIZ_DIR/pipelines/production/lsstcam_usdf/lsstCamScienceSensorUSDF_Danish.yaml#step1a-detectors,step1b-visits
    payload:
      butlerConfig: "/repo/embargo"
      payloadName: "aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315"
      inCollection: "LSSTCam/defaults,u/gmegias/intrinsic_aberrations_collection_temp"
      dataQuery: "instrument='LSSTCam' and exposure.observation_type in ('cwfs') and detector.purpose in ('SCIENCE') and exposure.day_obs in (20260315) and band in ('u', 'g', 'r', 'i', 'z
    ', 'y')"
    clusterAlgorithm: lsst.ctrl.bps.quantum_clustering_funcs.dimension_clustering
    cluster:
      cluster1:
        pipetasks: isr, generateDonutDirectDetectTask
        dimensions: detector
        equalDimensions: exposure:visit
        partitionDimensions: exposure
        partitionMaxClusters: 10000


Wondering whether it contains just `BLOCK-T614_triplets` or something else.  It seems that I should be able to constrain the input  on the block name 

In [7]:
from lsst.daf import butler as dafButler
butler = dafButler.Butler('/repo/embargo')
#collections = 'LSSTCam/defaults,u/gmegias/intrinsic_aberrations_collection_temp'

# Check if the raws are there ?  p 
refs = list(butler.registry.queryDatasets(
    'raw',
    collections=['LSSTCam/defaults'], day_obs = 20260315,
    where="instrument='LSSTCam' \
    and exposure.observation_type in ('cwfs') \
    and detector.purpose in ('SCIENCE') \
    and exposure.day_obs in (20260315) \
    and band in ('u', 'g', 'r', 'i', 'z', 'y') "
).expanded())


In [8]:
len(refs)

27972

In [25]:
import numpy as np 
science_programs = np.unique([ref.dataId.exposure.science_program for ref in refs])

In [26]:
science_programs

array(['BLOCK-T614_triplets'], dtype='<U19')

Ok, how can it be that we're not constraining science program, yet the only thing is BLOCK-T614_triplets? 

In [ ]:
That block was run on 3/15, 3/17, 3/24, 3/25, 3/26,  3/27, 3/31, 4/1, 4/2 , 4/4,  4/9.

I just run  `bps_submit_fam_template_20260315.yaml`... Try combining subsequent few days:

instead of `bps_submit_fam_template_20260315.yaml` make 

`bps_submit_fam_template_20260317-20260409.yaml` with 



computeSite: s3df
pipelineYaml: $DONUT_VIZ_DIR/pipelines/production/lsstcam_usdf/lsstCamScienceSensorUSDF_Danish.yaml#step1a-detectors,step1b-visits
payload:
  butlerConfig: "/repo/embargo"
  payloadName: "aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315"
  inCollection: "LSSTCam/defaults,u/gmegias/intrinsic_aberrations_collection_temp"
  dataQuery: "instrument='LSSTCam' and exposure.observation_type in ('cwfs') and detector.purpose in ('SCIENCE') and exposure.day_obs in (20260315) and band in ('u', 'g', 'r', 'i', 'z
', 'y')"
clusterAlgorithm: lsst.ctrl.bps.quantum_clustering_funcs.dimension_clustering
cluster:
  cluster1:
    pipetasks: isr, generateDonutDirectDetectTask
    dimensions: detector
    equalDimensions: exposure:visit
    partitionDimensions: exposure
    partitionMaxClusters: 10000


Well, but it's true that if it chokes it may be hard... Let's run a python script that creates these. As it is, I'm running by hand 


lsst
setup lsst_sitcom
aos
setup -kr ~/link_to_scichris/WORK/aos_packages/danish/


nohup bash service.sh > service_step2.out &

python render_yaml.py 20260315  -t bps_submit_fam_template.yam -o bps_submit_fam_template_20260315.yaml
python render_yaml.py 20260315  -t bps_submit_fam_step2_template.yaml -o bps_submit_fam_step2_template_20260315.yaml


nohup bps submit bps_submit_fam_template_20260315.yaml > nohup_step1_20260315.out &
nohup bps submit bps_submit_fam_step2_template_20260315.yaml > nohup_step2_20260315.out &




ok I got a python script that calls these scripts ... First step1, then step2. I don't concurrently submit both as step2 may fail instead of waiting for step1 to finish..,. 

Also, I change to maxnodes 40 : 

nohup bash service_40.sh > service_step1_0317-0409.out &
nohup bash service_40_torino.sh > service_40_torino.out &

python run_fam_batch.py --step1-only --dry-run

looks ok, also added 30s sleep between each day_obs submission...


 python run_fam_batch.py --step1-only 


Then I do 

cd /sdf/group/rubin/shared/scichris/LSSTCam_FAM_reprocessing

tail -f nohup_step1_20260317.out 
tail -f nohup_step1_20260324.out 




nohup bash service_40.sh > service_step2_0317-0409.out &
python run_fam_batch.py --step2-only 

cd /sdf/group/rubin/shared/scichris/LSSTCam_FAM_reprocessing/


tail -f nohup_step1_20260315.out :  worked fine 
 nohup_step1_20260317.out : worked fine 


Run by hand step2 for 0315 

python render_yaml.py 20260315 -t bps_submit_fam_step2_template.yaml -o bps_submit_fam_step2_template_20260315.yaml
nohup bps submit bps_submit_fam_step2_template_20260315.yaml > nohup_step2_20260315.out 2>&1 &


    srun --account rubin:commissioning -q torino -n 1 -c 16 --mem=32G --time=01:00:00 \
  bash /sdf/data/rubin/shared/scichris/LSSTCam_FAM_reprocessing/submit/u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315/20260415T203534Z/final_job.bash \
  /sdf/data/rubin/shared/scichris/LSSTCam_FAM_reprocessing/submit/u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315/20260415T203534Z/u_scichris_aos_fam_danish_wep_v16_9_0_donut_viz_v3_6_1_20260315_20260415T203534Z.qg \
  /repo/embargo


ok : if finalJob gets stranded on any other time, we should 


        * check which nodes I have access to 

        sacctmgr show associations user=scichris format=account%30,partition%15

        * look at submit file to get arguments 

            cat /sdf/data/rubin/shared/scichris/LSSTCam_FAM_reprocessing/submit/u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315/20260415T203534Z/jobs/finalJob/finalJob.sub

                
        

Next:

- ensure that all expected data products are there
- chain new collections to `aos_fam_danish_v1_triplets_bin_1x` : do the first day manually, then all others with a python script that calls these all

Test each day for how many quanta we'd expect... 

step1 for 20260315:

    more nohup_step1_20260315.out : shows that it submitted just fine 
    bps report --id 31483523.0   shows  that all jobs succeeded but  the finalJob is still running...

     X   STATE   %S     ID     OPERATOR PROJECT CAMPAIGN                       PAYLOAD                                                              RUN                                       
    --- ------- --- ---------- -------- ------- -------- ---------------------------------------------------- --------------------------------------------------------------------------------
        RUNNING  99 31483523.0 scichris                  aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315 u_scichris_aos_fam_danish_wep_v16_9_0_donut_viz_v3_6_1_20260315_20260415T203534Z
    
    
    Path: /sdf/data/rubin/shared/scichris/LSSTCam_FAM_reprocessing/submit/u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315/20260415T203534Z
    Global job id: sdfiana014.sdf.slac.stanford.edu#31483523.0#1776287676
    
    
                                       UNKNOWN MISFIT UNREADY READY PENDING RUNNING DELETED HELD SUCCEEDED FAILED PRUNED EXPECTED
    ---------------------------------- ------- ------ ------- ----- ------- ------- ------- ---- --------- ------ ------ --------
    TOTAL                                    0      0       0     0       1       0       0    0     51787      0      0    51788
    ---------------------------------- ------- ------ ------- ----- ------- ------- ------- ---- --------- ------ ------ --------

    pipetaskInit                             0      0       0     0       0       0       0    0         1      0      0        1
    cluster1                                 0      0       0     0       0       0       0    0      9828      0      0     9828
    cutOutDonutsScienceSensorGroupTask       0      0       0     0       0       0       0    0     13986      0      0    13986
    calcZernikesTask                         0      0       0     0       0       0       0    0     27972      0      0    27972
    finalJob                                 0      0       0     0       1       0       0    0         0      0      0        1
    


In [ ]:
ok, all step2's  are stuck because step1 is not finished... 

 python check_step2_status.py 


Checking why step1 even on 3/15 is still pending on finalJob ... 

squeue -u scichris -t RUNNING  # shows which  nodes are running my glide_ins 




For 3/15 I know from `nohup_step1_20260315.out` that there should be 

    Quanta               Tasks               
    ------ ----------------------------------
     27972                                isr
     27972      generateDonutDirectDetectTask
     13986 cutOutDonutsScienceSensorGroupTask
     27972                   calcZernikesTask


In [30]:
from lsst.daf import butler as dafButler
butler = dafButler.Butler('/repo/embargo')
collection = 'u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/20260315'
for datasetType in ['post_isr_image', 'donutTable', 'donutStampsExtra', 'zernikes']:
    refs = list(butler.registry.queryDatasets(
        datasetType,
        collections=[collection], day_obs = 20260315,
        where="instrument='LSSTCam'"
    ).expanded())
    print(f'There are {len(refs)} of {datasetType}')


There are 27972 of post_isr_image
There are 27972 of donutTable
There are 13986 of donutStampsExtra
There are 13986 of zernikes


In [ ]:
ok, all worked! Now proceed to step2 ... 

nohup bps submit bps_submit_fam_step2_template_20260315.yaml > nohup_step2_20260315.out &


In [ ]:
srun --account rubin:developers -p milano -n 1 -c 4 --mem=16G --time=02:00:00 \
  pipetask run \
  -b /repo/embargo \
  -i "LSSTCam/defaults,u/gmegias/intrinsic_aberrations_collection_temp" \
  -o "u/scichris/debug_calcZernikes_20260324_det164" \
  -p "$DONUT_VIZ_DIR/pipelines/production/lsstcam_usdf/lsstCamScienceSensorUSDF_Danish.yaml#step1a-detectors,step1b-visits" \
  -d "instrument='LSSTCam' and detector=164 and exposure.seq_num in (528, 529) and exposure.day_obs=20260324" \
  --register-dataset-types

In [32]:
from lsst.daf import butler as dafButler
butler = dafButler.Butler('/repo/embargo')
day_obs = 20260324
collection = f'u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/{day_obs}'
for datasetType in ['post_isr_image', 'donutTable', 'donutStampsExtra', 'zernikes']:
    refs = list(butler.registry.queryDatasets(
        datasetType,
        collections=[collection], day_obs = day_obs,
        where="instrument='LSSTCam'"
    ).expanded())
    print(f'There are {len(refs)} of {datasetType}')


There are 19278 of post_isr_image
There are 19278 of donutTable
There are 9639 of donutStampsExtra
There are 9638 of zernikes


Yeah, because for 1 quantum (calcZernikesTask:{instrument: 'LSSTCam', detector: 164, visit: 2026032400529, band: 'i', day_obs: 20260324, physical_filter: 'i_39') it failed . 

In [ ]:
Start step2 for that day:  

    
    nohup bps submit bps_submit_fam_step2_template_20260324.yaml > nohup_step2_20260324.out &


Now that it's aggregated (bar the missing quanta) I check that the data products exist so we can proceed to step2: 

In [34]:
from lsst.daf import butler as dafButler
butler = dafButler.Butler('/repo/embargo')
day_obs = 20260326
collection = f'u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/{day_obs}'
for datasetType in ['post_isr_image', 'donutTable', 'donutStampsExtra', 'zernikes']:
    refs = list(butler.registry.queryDatasets(
        datasetType,
        collections=[collection], day_obs = day_obs,
        where="instrument='LSSTCam'"
    ).expanded())
    print(f'There are {len(refs)} of {datasetType}')


There are 2457 of post_isr_image
There are 2457 of donutTable
There are 1134 of donutStampsExtra
There are 1131 of zernikes


In [ ]:
Start step2 for that day: 

   nohup bps submit bps_submit_fam_step2_template_20260326.yaml > nohup_step2_20260326.out &



In [35]:
from lsst.daf import butler as dafButler
butler = dafButler.Butler('/repo/embargo')
day_obs = 20260404
collection = f'u/scichris/aos_fam_danish/wep_v16_9_0/donut_viz_v3_6_1/{day_obs}'
for datasetType in ['post_isr_image', 'donutTable', 'donutStampsExtra', 'zernikes']:
    refs = list(butler.registry.queryDatasets(
        datasetType,
        collections=[collection], day_obs = day_obs,
        where="instrument='LSSTCam'"
    ).expanded())
    print(f'There are {len(refs)} of {datasetType}')


There are 18522 of post_isr_image
There are 18522 of donutTable
There are 9261 of donutStampsExtra
There are 9256 of zernikes


In [ ]:
ok, looks like they're there, we can submit step2 : 

    nohup bps submit bps_submit_fam_step2_template_20260404.yaml > nohup_step2_20260404.out &


In [ ]:
So it looks like at least several are `pending` on the `finalJob`..  I